# PT-W3-D4 概念实验：Policy 约束 Agent 执行

Capability 说明能做什么，Policy 说明当前是否允许做。执行前依次检查 effect_policy、required_scopes、human_review_gate。

## 实验 1：三道 guard 检查链

对应 LangChat 的 `read_only / conditional_write`、scope 交集和人审门；任何一关失败都不能执行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

CAPABILITIES = {
    'read_contract': {'effect_policy': 'read_only', 'required_scopes': {'contract:read'}, 'human_review_gate': 'none'},
    'export_due_list': {'effect_policy': 'read_only', 'required_scopes': {'contract:read', 'report:export'}, 'human_review_gate': 'conditional'},
    'submit_termination': {'effect_policy': 'conditional_write', 'required_scopes': {'lease:terminate'}, 'human_review_gate': 'mandatory'},
}
print('策略声明:', CAPABILITIES)

In [ ]:
def policy_guard(capability, actor_scopes, human_approved=False):
    p = CAPABILITIES[capability]
    checks = []
    if p['effect_policy'] == 'read_only': checks.append(('effect_policy', True))
    else: checks.append(('effect_policy', human_approved))
    checks.append(('required_scopes', p['required_scopes'] <= set(actor_scopes)))
    gate_ok = p['human_review_gate'] == 'none' or human_approved
    checks.append(('human_review_gate', gate_ok))
    passed = all(ok for _, ok in checks)
    return {'decision': 'allow' if passed else 'deny', 'checks': checks}

scenarios = [
    ('read_contract', {'contract:read'}, False),
    ('export_due_list', {'contract:read'}, False),
    ('submit_termination', {'lease:terminate'}, False),
    ('submit_termination', {'lease:terminate'}, True),
]
for cap, scopes, approved in scenarios: print(cap, approved, '=>', policy_guard(cap, scopes, approved))

In [ ]:
decisions = [policy_guard(*s)['decision'] for s in scenarios]
allow = decisions.count('allow'); deny = decisions.count('deny')
print(f'放行={allow}，拦截={deny}')
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['allow', 'deny'], [allow, deny], color=['#2a9d8f', '#e76f51'])
ax.set_title('Policy guard 决策'); ax.set_ylabel('场景数')
plt.tight_layout(); plt.show()